# Joint families — MLP + Transformer (PyTorch)

Demonstrates the two torch-based estimator families for `MixedTypeMultiTargetScorer` on a synthetic dataset with four heterogeneous target types.

- `JointMultiTargetMLPEstimator` — shared MLP encoder + per-target heads.
- `JointMultiTargetTransformerEstimator` — FT-Transformer-style feature tokenization + CLS pooling.

**No xgboost / lightgbm in this notebook** — that's covered in [independent.ipynb](independent.ipynb). On macOS, mixing torch and lightgbm in the same kernel triggers an OpenBLAS+Accelerate collision; splitting by backend avoids it.

Conditional inference (real-time-label conditioning via `OBSERVED_*`) is in [conditional.ipynb](conditional.ipynb).

In [1]:
# ruff: noqa: E402  (thread-pool env vars must be set before imports)
# macOS BLAS/OpenMP guard — set BEFORE numpy/torch import.
import os

for _var in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "VECLIB_MAXIMUM_THREADS", "OPENBLAS_NUM_THREADS"):
    os.environ.setdefault(_var, "1")

import warnings

warnings.filterwarnings("ignore", category=UserWarning)

import numpy as np
import pandas as pd

from skrec.constants import USER_ID_NAME
from skrec.estimator.classification import (
    JointMultiTargetMLPEstimator,
    JointMultiTargetTransformerEstimator,
)
from skrec.evaluator.datatypes import RecommenderEvaluatorType
from skrec.metrics.datatypes import RecommenderMetricType
from skrec.orchestrator import (
    TargetGroupSpec,
    TargetType,
    capability_matrix,
)
from skrec.recommender.ranking.ranking_recommender import RankingRecommender
from skrec.scorer.mixed_type_multi_target import MixedTypeMultiTargetScorer

print("scikit-rec multi-target capabilities:")
cm = capability_matrix()
print("  multi_target_model_types  =", cm["multi_target_model_types"])
print("  target_types              =", cm["target_types"])
print("  target_type_metric_compat =", cm["target_type_metric_compat"])

scikit-rec multi-target capabilities:
  multi_target_model_types  = ('joint_mlp', 'joint_transformer', 'independent', 'conditional_joint_mlp', 'conditional_joint_transformer')
  target_types              = ('binary', 'regression', 'multiclass', 'multilabel')
  target_type_metric_compat = {'binary': ('roc_auc', 'pr_auc'), 'regression': ('rmse', 'mae'), 'multiclass': ('multiclass_accuracy',), 'multilabel': ('roc_auc', 'pr_auc')}


## 1. Synthetic data with four heterogeneous targets

5 features, 800 users, four targets of different types — same dataset shape used by the independent and conditional notebooks for direct comparison.

In [2]:
def make_synthetic(n=800, seed=42):
    rng = np.random.default_rng(seed)
    X = pd.DataFrame(rng.normal(size=(n, 5)), columns=[f"feat_{i}" for i in range(5)])
    y_binary = (X["feat_0"] > 0).astype(int).to_numpy()
    y_reg = (2.5 * X["feat_1"] + rng.normal(scale=0.2, size=n)).to_numpy()
    cls_idx = np.column_stack([X["feat_2"], X["feat_3"], X["feat_4"]]).argmax(axis=1)
    y_mc = np.array(["action_A", "action_B", "action_C"])[cls_idx]
    y_ml = np.column_stack(
        [
            (X["feat_2"] > 0).astype(int).to_numpy(),
            (X["feat_3"] > 0).astype(int).to_numpy(),
        ]
    )
    target_specs = {
        "ITEM_clicked": TargetType.BINARY,
        "ITEM_revenue": TargetType.REGRESSION,
        "ITEM_action": TargetType.MULTICLASS,
        "engagement": TargetGroupSpec(
            type=TargetType.MULTILABEL,
            columns=["ITEM_email_open", "ITEM_app_open"],
        ),
    }
    y = {
        "ITEM_clicked": y_binary,
        "ITEM_revenue": y_reg,
        "ITEM_action": y_mc,
        "engagement": y_ml,
    }
    return X, y, target_specs


X, y, target_specs = make_synthetic(n=800)
split = 600
X_train, X_valid = X.iloc[:split], X.iloc[split:].reset_index(drop=True)
y_train = {k: v[:split] for k, v in y.items()}
y_valid = {k: v[split:] for k, v in y.items()}
print("train:", X_train.shape, "  valid:", X_valid.shape)
print("targets:", list(target_specs.keys()))

train: (600, 5)   valid: (200, 5)
targets: ['ITEM_clicked', 'ITEM_revenue', 'ITEM_action', 'engagement']


## 2. Joint MLP estimator

Shared MLP encoder + per-target heads. Trains all four targets jointly via summed per-type losses (BCE for binary/multilabel, MSE for regression, cross-entropy for multiclass).

In [3]:
joint_mlp = JointMultiTargetMLPEstimator(
    target_specs=target_specs,
    params={"epochs": 10, "hidden_dim": 64, "num_layers": 2, "batch_size": 64, "seed": 0},
)
joint_mlp.fit(X_train, y_train, X_valid=X_valid, y_valid=y_valid)
print("joint_mlp trained.")

2026-05-26 04:49:22,040 - skrec.estimator.classification._joint_multi_target_base - INFO Epoch [1/10] - Train Loss: 0.8361


2026-05-26 04:49:22,041 - skrec.estimator.classification._joint_multi_target_base - INFO   Validation Loss: 0.7824


2026-05-26 04:49:22,045 - skrec.estimator.classification._joint_multi_target_base - INFO Epoch [2/10] - Train Loss: 0.7409


2026-05-26 04:49:22,046 - skrec.estimator.classification._joint_multi_target_base - INFO   Validation Loss: 0.6971


2026-05-26 04:49:22,050 - skrec.estimator.classification._joint_multi_target_base - INFO Epoch [3/10] - Train Loss: 0.6585


...

2026-05-26 04:49:22,079 - skrec.estimator.classification._joint_multi_target_base - INFO Epoch [9/10] - Train Loss: 0.3618


2026-05-26 04:49:22,079 - skrec.estimator.classification._joint_multi_target_base - INFO   Validation Loss: 0.3320


2026-05-26 04:49:22,084 - skrec.estimator.classification._joint_multi_target_base - INFO Epoch [10/10] - Train Loss: 0.3299


2026-05-26 04:49:22,084 - skrec.estimator.classification._joint_multi_target_base - INFO   Validation Loss: 0.2988


joint_mlp trained.


## 3. Joint Transformer estimator

FT-Transformer-style: each input feature is tokenized into a `d_model`-dim embedding, a learnable CLS token is prepended, and the sequence passes through `n_layers` Transformer encoder blocks. CLS pooling produces the final hidden representation that feeds the per-target heads.

In [4]:
joint_tx = JointMultiTargetTransformerEstimator(
    target_specs=target_specs,
    params={
        "epochs": 10,
        "d_model": 32,
        "n_heads": 4,
        "n_layers": 2,
        "ffn_dim": 64,
        "batch_size": 64,
        "seed": 0,
    },
)
joint_tx.fit(X_train, y_train, X_valid=X_valid, y_valid=y_valid)
print("joint_transformer trained.")

2026-05-26 04:49:22,115 - skrec.estimator.classification._joint_multi_target_base - INFO Epoch [1/10] - Train Loss: 0.8964


2026-05-26 04:49:22,117 - skrec.estimator.classification._joint_multi_target_base - INFO   Validation Loss: 0.8500


2026-05-26 04:49:22,141 - skrec.estimator.classification._joint_multi_target_base - INFO Epoch [2/10] - Train Loss: 0.8288


2026-05-26 04:49:22,143 - skrec.estimator.classification._joint_multi_target_base - INFO   Validation Loss: 0.7616


2026-05-26 04:49:22,167 - skrec.estimator.classification._joint_multi_target_base - INFO Epoch [3/10] - Train Loss: 0.7353


...

2026-05-26 04:49:22,325 - skrec.estimator.classification._joint_multi_target_base - INFO Epoch [9/10] - Train Loss: 0.3107


2026-05-26 04:49:22,327 - skrec.estimator.classification._joint_multi_target_base - INFO   Validation Loss: 0.2311


2026-05-26 04:49:22,351 - skrec.estimator.classification._joint_multi_target_base - INFO Epoch [10/10] - Train Loss: 0.2591


2026-05-26 04:49:22,352 - skrec.estimator.classification._joint_multi_target_base - INFO   Validation Loss: 0.1858


joint_transformer trained.


## 4. Per-target evaluation — `Dict[str, float]`

`MixedTypeMultiTargetScorer.evaluate()` always returns a per-target dict. Heterogeneous types can't be macro-averaged; each declared `TargetType` gets its appropriate metric.

In [5]:
def build_recommender(estimator):
    scorer = MixedTypeMultiTargetScorer(
        estimator=estimator,
        target_specs=target_specs,
    )
    return RankingRecommender(scorer=scorer)


valid_inf = X_valid.copy()
valid_inf.insert(0, USER_ID_NAME, [f"u_{i}" for i in range(len(X_valid))])

# Wide-format logged_rewards matches predict_targets's output column set.
logged = pd.DataFrame(
    {
        "ITEM_clicked": y_valid["ITEM_clicked"],
        "ITEM_revenue": y_valid["ITEM_revenue"],
        "ITEM_action": y_valid["ITEM_action"],
        "ITEM_email_open": y_valid["engagement"][:, 0],
        "ITEM_app_open": y_valid["engagement"][:, 1],
    }
)

per_target_metrics = {
    "ITEM_clicked": RecommenderMetricType.ROC_AUC,
    "ITEM_revenue": RecommenderMetricType.MAE,
    "ITEM_action": RecommenderMetricType.MULTICLASS_ACCURACY,
    "ITEM_email_open": RecommenderMetricType.ROC_AUC,
    "ITEM_app_open": RecommenderMetricType.ROC_AUC,
}

rows = []
for name, est in [("joint_mlp", joint_mlp), ("joint_transformer", joint_tx)]:
    rec = build_recommender(est)
    out = rec.evaluate(
        eval_type=RecommenderEvaluatorType.SIMPLE,
        metric_type=per_target_metrics,
        eval_top_k=10,
        score_items_kwargs={"interactions": valid_inf},
        eval_kwargs={"logged_rewards": logged},
    )
    rows.append({"family": name, **out})

comparison = pd.DataFrame(rows).set_index("family").round(4)
comparison

,ITEM_clicked,ITEM_revenue,ITEM_action,ITEM_email_open,ITEM_app_open
family,,,,,
joint_mlp,0.9619,0.3011,0.88,0.9850,0.9552
joint_transformer,0.9982,0.4768,0.92,0.9968,0.9876


## 5. `score_per_target` — user-supplied metrics

For metrics outside the named v2 set (log-loss, macro-F1, business-specific), use `score_per_target` with sklearn callables. Name-keyed entries override `TargetType`-keyed defaults.

In [6]:
from sklearn.metrics import f1_score, log_loss, mean_absolute_percentage_error

scorer_mlp = MixedTypeMultiTargetScorer(estimator=joint_mlp, target_specs=target_specs)
user_metrics = scorer_mlp.score_per_target(
    interactions=valid_inf,
    y_true=logged,
    metric_callables={
        TargetType.BINARY: lambda y_true, p: float(log_loss(y_true, p[:, 1], labels=[0, 1])),
        TargetType.REGRESSION: lambda y_true, p: float(
            mean_absolute_percentage_error(np.clip(np.abs(y_true), 1e-6, None), p)
        ),
        TargetType.MULTICLASS: lambda y_true, p: float(
            f1_score(
                [["action_A", "action_B", "action_C"].index(v) for v in y_true],
                p.argmax(axis=1),
                average="macro",
            )
        ),
    },
)
print("User-supplied metrics on joint_mlp predictions:")
for k, v in user_metrics.items():
    print(f"  {k:20s} = {v:.4f}")

User-supplied metrics on joint_mlp predictions:
  ITEM_clicked         = 0.3536
  ITEM_revenue         = 1.3017
  ITEM_action          = 0.8792
  ITEM_email_open      = 0.3973
  ITEM_app_open        = 0.3872


## 6. `recommend()` short-circuit

`recommend()` on `MixedTypeMultiTargetScorer` short-circuits to per-target point estimates (it's not a ranking). `top_k` emits a warning.

In [7]:
preds = build_recommender(joint_mlp).recommend(interactions=valid_inf.head(5))
print("predict_targets output (first 5 rows):")
preds.round(3)

predict_targets output (first 5 rows):


,ITEM_clicked,ITEM_revenue,ITEM_action,ITEM_email_open,ITEM_app_open
0,1,1.204,action_A,1,0
1,0,-4.241,action_A,1,0
2,1,-0.621,action_C,0,1
3,0,-1.654,action_A,1,0
4,0,3.474,action_A,1,0


## Where to go next

- **Independent (xgboost / lightgbm)**: [independent.ipynb](independent.ipynb) — per-target sub-estimators with tree-based learners
- **Conditional inference (v3)**: [conditional.ipynb](conditional.ipynb) — real-time-label conditioning via `OBSERVED_*`
- **Decision rule**: [docs/user-guide/decision-rule.md](../../docs/user-guide/decision-rule.md)
- **Scorer reference**: [docs/user-guide/scorers.md](../../docs/user-guide/scorers.md#5-mixedtypemultitargetscorer)